# TARANG: Edge Coupler Module Examples

In [ ]:
# ---- Logging ---- #
from tarang import configure_logging, set_tidy3d_output

configure_logging(level="INFO")  # Show programmer-written TARANG logs
set_tidy3d_output(False)         # Suppress Tidy3D chatter and managed progress bars

# ---- Define Media ---- #
from tarang.media import get_medium, get_medium_dispersion

# 1) PECDV SiN Data
Si3N4 = get_medium(
    medium_key = "si3n4",
    wl_range = (1.550 - 0.200, 1.550 + 0.200),
    tolerance = 1e-5
)

# 2) Silica
SiO2 = get_medium(
    medium_key = "sio2",
    wl_range = (1.550 - 0.200, 1.550 + 0.200),
    tolerance = 5e-5
)

## Edge Coupler: Example Geometry

In [ ]:
# ---- Define an Arbitrary Geometry for an Out-Copupling Edge Coupler---- #
from tarang.geometries import edge_coupler

example_edge_out_coupler = edge_coupler.get_active_simulation(
    wavelength_um = 1.550,
    
    wg_width = 0.800,
    wg_height = 0.800,

    core_medium = Si3N4,
    box_medium = SiO2,

    tip_width = 0.150,
    taper_length = 30,      # We keep the numbers low so we can actually see it!

    out_coupling = True,    # Declare out-coupling behavior

    fine_steps_per_wvl = 20,
    coarse_steps_per_wvl = 15


)

# Visualize 3D Geometry
from tarang.utilities.sim_plotter import display_3d

display_3d(
    device = example_edge_out_coupler,
    output_path = "example_results/edge_coupler/example_edge__out_coupler.png",
    save = True,
    coordinates = edge_coupler.get_optimal_plot_coords(example_edge_out_coupler.sim)
)

In [ ]:
# ---- Define an Arbitrary Geometry for an In-Copupling Edge Coupler---- #
from tarang.geometries import edge_coupler

example_edge_in_coupler = edge_coupler.get_active_simulation(
    wavelength_um = 1.550,
    
    wg_width = 0.800,
    wg_height = 0.800,

    core_medium = Si3N4,
    box_medium = SiO2,

    tip_width = 0.150,
    taper_length = 30,  # We keep the numbers low so we can actually see it!

    in_coupling = True # Declare in-coupling behavior
)

# Visualize 3D Geometry
from tarang.utilities.sim_plotter import display_3d

display_3d(
    device = example_edge_in_coupler,
    output_path = "example_results/edge_coupler/example_edge__out_coupler.png",
    save = True,
    coordinates = edge_coupler.get_optimal_plot_coords(example_edge_in_coupler.sim)
)

## Edge Coupler Methods

In [ ]:
# ---- Compute and extract mode coupling efficiency ---- #
output_dir = "example_results/edge_coupler"

efficiency = example_edge_out_coupler.get_efficiency(
    output_dir=output_dir,
    verbose=True
)

print(f"\nCalculated Coupling Efficiency: {efficiency * 100.0:.2f}%")

In [ ]:
# ---- Extract Insertion Loss in dB ---- #
loss_db = example_edge_out_coupler.get_loss_db(
    output_dir=output_dir,
    verbose=False
)

print(f"Calculated Insertion Loss: {loss_db:.2f} dB")

In [ ]:
# ---- Plot spatial field intensity distributions across XY and XZ cross-sections ---- #
# XY Plane (Top-down view along propagation axis)
example_edge_out_coupler.plot_fields(
    output_dir=output_dir,
    monitor_name="field_xy",
    save=True,
    show=True
)

# XZ Plane (Side cross-section through core center)
example_edge_out_coupler.plot_fields(
    output_dir=output_dir,
    monitor_name="field_xz",
    save=True,
    show=True
)

### Edge-Coupler Optimization

In [ ]:
# ---- 2D Parametric Optimization Sweep (Out-Coupling) ---- #
from tarang.edge_coupler import parametric_optimization_sweep
import numpy as np

# Small 3x3 parameter grid with coarse meshing for a quick test run
sweep_taper_lengths = [10.0, 20.0, 30.0, 40.0, 50.0]  # μm
sweep_tip_widths = [0.100, 0.150, 0.200, 0.250, 0.300]  # μm

efficiency_matrix = parametric_optimization_sweep(
    sweep_taper_lengths=sweep_taper_lengths,
    sweep_tip_widths=sweep_tip_widths,
    output_dir="example_results/edge_coupler/edge_coupler_sweep",
    
    wg_width=0.800,
    wg_height=0.800,
    core_medium=Si3N4,
    box_medium=SiO2,
    wavelength_um=1.550,
    
    out_coupling=True,
    fine_steps_per_wvl=15,
    coarse_steps_per_wvl=10,
    
    batching=True,
    show=True,
    save=True
)

print("\nEfficiency Matrix (%):")
print(np.round(efficiency_matrix * 100.0, 2))

In [ ]:
# ---- Test the Optimized Edge Coupler ---- #
from pathlib import Path
from tarang.geometries import edge_coupler

results_dir = Path("example_results/edger_coupler/optimized_edge_coupler/")

optimized_edge_coupler = edge_coupler.get_active_simulation(
    wavelength_um = 1.550,
    
    wg_width = 0.800,
    wg_height = 0.800,

    core_medium = Si3N4,
    box_medium = SiO2,

    # Use the numbers from the previous cell
    tip_width = 0.250,
    taper_length = 20,

    out_coupling = True,    # Declare out-coupling behavior

    # Meshing (higher resolution this time)
    fine_steps_per_wvl = 20,
    coarse_steps_per_wvl = 15
)

# Get efficiency (high resolution simulation)
optimized_efficiency : float = optimized_edge_coupler.get_efficiency(
    output_dir = results_dir,
    verbose = True
)

# Get loss
optimized_loss_db : float = optimized_edge_coupler.get_loss_db(
    output_dir = results_dir,
    verbose = True
)

# Get field plots
optimized_edge_coupler.plot_fields(
    output_dir=results_dir,
    monitor_name="field_xz",
    save=True,
    show=True
)

# Print Results
print(f"\nCalculated Coupling Efficiency: {optimized_efficiency * 100.0:.2f}%")
print(f"Calculated Insertion Loss: {optimized_loss_db:.2f} dB")